In [1]:
import subprocess

try:
    subprocess.run(
        ["git", "clone", "https://github.com/shatyamsharma7416-oss/book_recommendation.git"],
        check=True
    )
    print("Repository cloned successfully!")
except subprocess.CalledProcessError as e:
    print(f"An error occurred while cloning: {e}")

Repository cloned successfully!


In [2]:
import numpy as np
import pandas as pd

[Books](https://raw.githubusercontent.com/shatyamsharma7416-oss/book_recommendation/refs/heads/main/dataset/Books.csv)

[Ratings](https://raw.githubusercontent.com/shatyamsharma7416-oss/book_recommendation/refs/heads/main/dataset/Ratings.csv)

[Users](https://raw.githubusercontent.com/shatyamsharma7416-oss/book_recommendation/refs/heads/main/dataset/Users.csv)

In [3]:
books = pd.read_csv("book_recommendation/dataset/Books.csv")
ratings = pd.read_csv("book_recommendation/dataset/Ratings.csv")
users = pd.read_csv("book_recommendation/dataset/Users.csv")

/tmp/ipykernel_7298/3201147768.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("book_recommendation/dataset/Books.csv")
/tmp/ipykernel_7298/3201147768.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  ratings = pd.read_csv("book_recommendation/dataset/Ratings.csv")
/tmp/ipykernel_7298/3201147768.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("book_recommendation/dataset/Users.csv")


In [ ]:
books.head(4)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...


In [ ]:
ratings.head(4)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0


In [ ]:
users.head(4)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0


In [ ]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271360, 8)
(1149781, 3)
(278859, 3)


### Null values

In [ ]:
books.isna().sum()

,0
ISBN,0
Book-Title,0
Book-Author,2
Year-Of-Publication,0
Publisher,2
Image-URL-S,0
Image-URL-M,0
Image-URL-L,3


In [ ]:
ratings.isna().sum()

,0
User-ID,0
ISBN,1
Book-Rating,1


In [ ]:
users.isna().sum()

,0
User-ID,0
Location,1
Age,110763


### Duplicate

In [11]:
books.duplicated().sum()

np.int64(0)

In [12]:
ratings.duplicated().sum()

np.int64(0)

In [13]:
users.duplicated().sum()

np.int64(0)

In [14]:
books_rating = books.iloc[:,:2].merge(ratings, on='ISBN', how='left')
books_rating.columns

Index(['ISBN', 'Book-Title', 'User-ID', 'Book-Rating'], dtype='object')

In [ ]:
def popular_book(dataframe):
  """
  Books that atleast rated by 100 readers
  and average rating is above 4.0, we
  will consider such books popular
  """
  # filter books that is read by more than 100 readers
  book_count = books_rating['ISBN'].value_counts().reset_index()
  book_count = book_count[book_count['count']>100]['ISBN']
  most_readed_books = books_rating[books_rating['ISBN'].isin(book_count)]

  # further filter books than has average rating more than 4
  high_rated_books  = most_readed_books.groupby("ISBN")['Book-Rating'].mean().reset_index()
  high_rated_books = high_rated_books[high_rated_books['Book-Rating']>4]

  return pd.merge(books.iloc[:,:2], high_rated_books, how='inner', on='ISBN')


In [16]:
famous_books = popular_book(books_rating).sort_values('Book-Rating', ascending=False).reset_index(drop=True)
famous_books

,ISBN,Book-Title,Book-Rating
0,0439064864,Harry Potter and the Chamber of Secrets (Book 2),6.611765
1,0439139597,Harry Potter and the Goblet of Fire (Book 4),6.541237
2,0439136350,Harry Potter and the Prisoner of Azkaban (Book 3),6.467005
3,0590353403,Harry Potter and the Sorcerer's Stone (Book 1),6.363095
4,043935806X,Harry Potter and the Order of the Phoenix (Boo...,5.571856
...,...,...,...
95,0842329129,Left Behind: A Novel of the Earth's Last Days ...,4.026936
96,0312990456,One for the Money (A Stephanie Plum Novel),4.021127
97,0385512104,The Curious Incident of the Dog in the Night-T...,4.017857
98,0140042598,On the Road,4.009901


In [ ]:
def regular_readers(dataframe):
  """
  Readers that rated atleast 200 books
  """
  user_count = books_rating['User-ID'].value_counts().reset_index()
  selected_users = user_count[user_count['count']>200]

  return selected_users['User-ID']


In [ ]:
often_readers = regular_readers(books_rating)
often_readers = often_readers.to_numpy()

In [ ]:
filtered_rating = books_rating[books_rating['User-ID'].isin(often_readers)]

In [20]:
well_known_books = filtered_rating.groupby("Book-Title").count()['Book-Rating'] > 50
well_known_books = well_known_books[well_known_books].index

In [21]:
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(well_known_books)]

In [ ]:
final_ratings

,ISBN,Book-Title,User-ID,Book-Rating
31,0399135782,The Kitchen God's Wife,11676,9.0
33,0399135782,The Kitchen God's Wife,36836,0.0
34,0399135782,The Kitchen God's Wife,46398,9.0
38,0399135782,The Kitchen God's Wife,113270,0.0
39,0399135782,The Kitchen God's Wife,113519,0.0
...,...,...,...,...
1029536,1878702831,Echoes,238781,0.0
1029728,0394429869,I Know Why the Caged Bird Sings,239594,8.0
1029730,0449001164,The Promise,239594,7.0
1029943,0743527631,The Pillars of the Earth,240144,0.0


In [23]:
pt = final_ratings.pivot_table(columns='User-ID', values='Book-Rating', index='Book-Title')

In [ ]:
pt.fillna(0, inplace=True)

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity_score = cosine_similarity(pt)

In [104]:
def recommend(book_name):
    #index fetch
    index = np.where(pt.index==book_name)[0][0]
    similar_items = sorted(list(enumerate(similarity_score[index])), key=lambda x:x[1], reverse=True)[1:6]

    for i in similar_items:
        print(pt.index[i[0]])

In [105]:
recommend("Harry Potter and the Chamber of Secrets (Book 2)")

Harry Potter and the Prisoner of Azkaban (Book 3)
Harry Potter and the Goblet of Fire (Book 4)
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))
Harry Potter and the Sorcerer's Stone (Book 1)
Harry Potter and the Order of the Phoenix (Book 5)


'The Hobbit The Enchanting Prelude To The Lord Of The Rings'